# Finz Analysis Model — Training on Colab

**GPU**: Runtime → Change runtime type → T4 GPU

이 노트북은 다음을 수행합니다:
1. 레포 클론 & 의존성 설치
2. 뉴스 + 주가 데이터 수집 (46개 종목, 15년치)
3. 모델 학습 (DistilBERT + Cross-Attention Fusion)
4. 학습된 모델 다운로드

## 0. GPU 확인

In [1]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU가 없습니다! Runtime → Change runtime type → T4 GPU 를 선택하세요.")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## 1. 레포 클론 & 설치

In [2]:
!git clone https://github.com/IkhyunKwon-00/finzanalysismodel.git
%cd finzanalysismodel
!pip install -e . -q

Cloning into 'finzanalysismodel'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 64 (delta 20), reused 57 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 47.95 KiB | 4.00 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/finzanalysismodel
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for finz-analysis-model (pyproject.toml) ... done


## 2. 데이터 수집

46개 종목 뉴스 + 주가 / 거시경제 데이터를 수집합니다.

- **Yahoo Finance** : 46개 종목 최신 뉴스 (자동)
- **GDELT** : S&P500 · 금 관련 과거 기사, 2015~현재 (자동, API key 불필요)
- **Financial PhraseBank** : 선택, 아래 셀 실행
- **Kaggle** : 선택, `--kaggle` 플래그로 ~4만 건+ 추가 수집 (아래 셀 참조)

In [3]:
# Yahoo Finance + GDELT (S&P500·Gold 과거 기사) 자동 포함
!python -m scripts.collect_data

[1/4] Collecting news from external sources...
Yahoo Finance news: 100% 44/44 [00:04<00:00,  9.42it/s]
Yahoo Finance: 440 articles (44 symbols)
Financial PhraseBank not found at data/raw/financial_phrasebank.txt
  → Download: https://huggingface.co/datasets/financial_phrasebank
No CSV files found in data/raw
  GDELT warning (^GSPC [2015-02-19–2015-05-19]): Expecting value: line 1 column 1 (char 0)
  GDELT warning (^GSPC [2015-05-19–2015-08-19]): HTTP Error 429: Too Many Requests
  GDELT warning (^GSPC [2015-08-19–2015-11-19]): HTTP Error 429: Too Many Requests
  GDELT warning (^GSPC [2015-11-19–2016-02-19]): HTTP Error 429: Too Many Requests
  GDELT warning (^GSPC [2016-02-19–2016-05-19]): HTTP Error 429: Too Many Requests
  GDELT warning (^GSPC [2016-05-19–2016-08-19]): HTTP Error 429: Too Many Requests
  GDELT warning (^GSPC [2016-08-19–2016-11-19]): HTTP Error 429: Too Many Requests
  GDELT warning (^GSPC [2016-11-19–2017-02-19]): HTTP Error 429: Too Many Requests
  GDELT warning (^

### (선택) Kaggle 금융 뉴스 다운로드 — 샘플 수 대폭 증가

Kaggle API 자격증명이 있으면 아래 셀을 실행하세요. 없으면 건너뛰어도 됩니다.

- `aaron7sun/stocknews` : Dow Jones 헤드라인 2008~2016 (~45,000 건)
- `miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests` : 종목 태그 포함 ~400만 건

Kaggle API 키 발급: https://www.kaggle.com/settings → "Create New Token"

In [4]:
import os

# Kaggle 자격증명 입력 (또는 ~/.kaggle/kaggle.json 이미 있으면 건너뜀)
KAGGLE_USERNAME = "ikhyunkwon"   # ← 본인 Kaggle 유저명
KAGGLE_KEY      = "KGAT_b8d3e6be992d5b169420b4afe6205b04"   # ← Kaggle API 키

if KAGGLE_USERNAME and KAGGLE_KEY:
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

!pip install kaggle -q

# Kaggle 데이터셋 다운로드 후 데이터 재수집 (GDELT 포함)
!python -m scripts.collect_data --kaggle

Dataset URL: https://www.kaggle.com/datasets/aaron7sun/stocknews
100% 5.82M/5.82M [00:01<00:00, 3.34MB/s]

  → Extracted to data/raw/
Dataset URL: https://www.kaggle.com/datasets/miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests
100% 210M/210M [00:12<00:00, 17.2MB/s]

  → Extracted to data/raw/
[1/4] Collecting news from external sources...
Yahoo Finance news: 100% 44/44 [00:04<00:00, 10.41it/s]
Yahoo Finance: 440 articles (44 symbols)
Financial PhraseBank not found at data/raw/financial_phrasebank.txt
  → Download: https://huggingface.co/datasets/financial_phrasebank
  Skipping Combined_News_DJIA.csv: no recognisable text column
  Skipping RedditNews.csv: no recognisable text column
CSV analyst_ratings_processed.csv: 1400468 rows
CSV raw_analyst_ratings.csv: 1407328 rows
CSV raw_partner_headlines.csv: 1845559 rows
  Skipping upload_DJIA_table.csv: no recognisable text column
  GDELT warning (^GSPC [2015-02-19–2015-05-19]): Expecting value: line 1 column 1 (char 0)
  GDELT w

### (선택) Financial PhraseBank 추가

학습 데이터를 ~5,000개 전문가 라벨링 문장으로 보강합니다.

In [5]:
# Financial PhraseBank 다운로드 (선택사항 — 학습 데이터 보강)
# Hugging Face에서 다운받아 data/raw/에 배치
try:
    from datasets import load_dataset
    ds = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)
    with open("data/raw/financial_phrasebank.txt", "w") as f:
        for row in ds["train"]:
            label_map = {0: "negative", 1: "neutral", 2: "positive"}
            f.write(f"{row['sentence']}@{label_map[row['label']]}\n")
    print(f"Financial PhraseBank saved: {len(ds['train'])} sentences")
    # 데이터 재수집 (PhraseBank 포함)
    !python -m scripts.collect_data
except Exception as e:
    print(f"PhraseBank 다운로드 실패 (선택사항이므로 무시 가능): {e}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'financial_phrasebank' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'financial_phrasebank' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able t

README.md: 0.00B [00:00, ?B/s]

financial_phrasebank.py: 0.00B [00:00, ?B/s]

PhraseBank 다운로드 실패 (선택사항이므로 무시 가능): Dataset scripts are no longer supported, but found financial_phrasebank.py


## 3. 데이터셋 확인

In [6]:
import pandas as pd

df = pd.read_parquet("data/processed/dataset.parquet")
print(f"총 샘플: {len(df)}")
print(f"컬럼: {list(df.columns)}")
print(f"\n종목 분포:")
print(df["symbol"].value_counts().head(20))
print(f"\n레이블 분포 (30d):")
print(df["label_30d"].value_counts())
df.head(3)

총 샘플: 1078
컬럼: ['symbol', 'published_at', 'title_en', 'summary_en', 'topic_id', 'source', 'return_30d', 'label_30d', 'return_180d', 'label_180d', 'return_360d', 'label_360d', 'stock_return_5d', 'stock_vol_5d', 'stock_return_20d', 'stock_vol_20d', 'stock_return_60d', 'stock_vol_60d', 'sp500_close', 'gold_close', 'wti_close', 'bitcoin_close', 'vix_close', 'dxy_close']

종목 분포:
symbol
GC=F     685
NVDA      10
GOOGL     10
AAPL      10
TSLA      10
IREN      10
ASML      10
SMR       10
ENPH      10
V         10
BRK-B     10
JPM       10
IONQ      10
MCHP      10
ADI       10
TXN       10
LIN       10
RTX       10
MCD       10
PFE       10
Name: count, dtype: int64

레이블 분포 (30d):
label_30d
1    716
2    202
0    160
Name: count, dtype: int64


,symbol,published_at,title_en,summary_en,topic_id,source,return_30d,label_30d,return_180d,label_180d,...,stock_return_20d,stock_vol_20d,stock_return_60d,stock_vol_60d,sp500_close,gold_close,wti_close,bitcoin_close,vix_close,dxy_close
0,TSLA,2026-03-18T14:30:43Z,BMW's new i3 EV takes on Tesla with 440-mile r...,BMW officially unveiled the new i3 EV on Wedne...,symbol:TSLA,Yahoo Finance,-6.319061,0,-6.319061,0,...,-0.042658,0.018940,-0.194430,0.020766,6624.700195,4889.899902,96.320000,71245.578125,25.090000,100.089996
1,TSLA,2026-03-22T06:26:38Z,"Musk announces ""Terafab"" to internalize Tesla ...",Investing.com -- Elon Musk has announced the l...,symbol:TSLA,Investing.com,0.000000,1,0.000000,1,...,-0.108095,0.021006,-0.269811,0.021256,6506.479980,4574.899902,98.230003,68711.523438,26.780001,99.650002
2,TSLA,2026-03-22T05:25:00Z,1 Tesla Competitor That Could Unseat the EV Gi...,Rivian's R2 should compete heavily with Tesla'...,symbol:TSLA,Motley Fool,0.000000,1,0.000000,1,...,-0.108095,0.021006,-0.269811,0.021256,6506.479980,4574.899902,98.230003,68711.523438,26.780001,99.650002


## 4. 모델 학습

T4 GPU 기준 설정:
- batch_size: 32
- max_length: 512
- freeze_layers: 4 (DistilBERT 6개 레이어 중 4개 동결)
- epochs: 30 (early stopping patience=5)

In [7]:
!python -m src.train --data data/processed/dataset.parquet

Device: cuda
Loading datasets...
config.json: 100% 483/483 [00:00<00:00, 1.52MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 184kB/s]
vocab.txt: 232kB [00:00, 860kB/s]
tokenizer.json: 466kB [00:00, 1.64MB/s]
Train: 808 | Val: 162
Numerical features: 12
model.safetensors: 100% 268M/268M [00:02<00:00, 92.8MB/s]
Loading weights: 100% 100/100 [00:00<00:00, 1073.61it/s, Materializing param=transformer.layer.5.sa_layer_norm.weight]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Parameters: 67,337,484 total, 38,985,996 trainable

═══ Epoch 1/30 ═══
  Train L

## 5. 학습 결과 확인

In [8]:
import torch
from pathlib import Path

best_path = Path("models/best/model.pt")
if best_path.exists():
    ckpt = torch.load(best_path, map_location="cpu", weights_only=False)
    print(f"Best model — Epoch: {ckpt['epoch']}")
    print(f"Val Loss: {ckpt['val_loss']:.4f}")
    if 'val_metrics' in ckpt:
        for k, v in ckpt['val_metrics'].items():
            print(f"  {k}: {v:.4f}")
else:
    print("모델 파일이 없습니다. 학습이 완료되었는지 확인하세요.")

Best model — Epoch: 7
Val Loss: 1.9728
  val_loss: 1.9728
  acc_horizon_0: 0.9568
  acc_horizon_1: 0.8704
  acc_horizon_2: 0.8889


## 6. 추론 테스트

In [9]:
import torch
from transformers import AutoTokenizer
from src.config import load_config
from src.models.momentum_model import FinzMomentumModel

cfg = load_config()
ckpt = torch.load("models/best/model.pt", map_location="cpu", weights_only=False)

model = FinzMomentumModel(
    num_numerical_features=ckpt["num_numerical_features"],
    text_model_name=cfg["text_encoder"]["model_name"],
    text_embed_dim=cfg["model"]["text_embed_dim"],
    numerical_hidden=cfg["model"]["numerical_hidden"],
    fusion_hidden=cfg["model"]["fusion_hidden"],
    num_heads=cfg["model"]["num_heads"],
    dropout=0.0,
    num_horizons=cfg["model"]["num_horizons"],
)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

tokenizer = AutoTokenizer.from_pretrained(cfg["text_encoder"]["model_name"])

# 테스트 뉴스
text = "Tesla reports record quarterly revenue on strong EV demand [SEP] Electric vehicle maker Tesla Inc reported record revenue of $25 billion in Q4."
enc = tokenizer(text, max_length=cfg["text_encoder"]["max_length"], padding="max_length", truncation=True, return_tensors="pt")
numerical = torch.zeros(1, ckpt["num_numerical_features"])

with torch.no_grad():
    outputs = model(enc["input_ids"], enc["attention_mask"], numerical)

LABELS = {0: "negative", 1: "neutral", 2: "positive"}
horizons = [30, 180, 360]
for i, h in enumerate(horizons):
    logits = outputs[f"logits_{i}"][0]
    probs = torch.softmax(logits, dim=-1)
    pred = probs.argmax().item()
    ret = outputs[f"return_{i}"][0].item()
    print(f"  {h}d: {LABELS[pred]} (conf={probs[pred].item()*100:.1f}%) | return={ret:.2f}%")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  30d: neutral (conf=60.4%) | return=-0.60%
  180d: negative (conf=51.6%) | return=-0.46%
  360d: neutral (conf=46.6%) | return=-0.29%


## 7. Sentence Attribution (핵심 문장 선별)

In [10]:
from src.models.sentence_attribution import get_sentence_attributions

text = "Tesla reports record quarterly revenue on strong EV demand. Electric vehicle maker Tesla Inc reported record revenue of $25 billion in Q4. The company also announced plans to expand its Gigafactory in Texas."
enc = tokenizer(text, max_length=cfg["text_encoder"]["max_length"], padding="max_length", truncation=True, return_tensors="pt")
numerical = torch.zeros(1, ckpt["num_numerical_features"])

results = get_sentence_attributions(
    model=model,
    tokenizer=tokenizer,
    text=text,
    input_ids=enc["input_ids"],
    attention_mask=enc["attention_mask"],
    numerical=numerical,
    target_horizon=0,
    n_steps=30,
    top_k=5,
)

print("핵심 문장 (Integrated Gradients 기반):")
for r in results:
    print(f"  #{r['rank']} (score={r['attribution_score']:.1f}) {r['sentence']}")

핵심 문장 (Integrated Gradients 기반):
  #1 (score=100.0) Tesla reports record quarterly revenue on strong EV demand.
  #2 (score=80.2) The company also announced plans to expand its Gigafactory in Texas.
  #3 (score=71.5) Electric vehicle maker Tesla Inc reported record revenue of $25 billion in Q4.


## 8. 모델 다운로드

학습된 모델을 로컬로 다운로드합니다.

In [11]:
from google.colab import files

# 최적 모델 다운로드
files.download("models/best/model.pt")
print("✅ model.pt 다운로드 완료 — 이 파일을 finzanalysismodel/models/best/ 에 배치하세요.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ model.pt 다운로드 완료 — 이 파일을 finzanalysismodel/models/best/ 에 배치하세요.


### (선택) Google Drive에 저장

In [13]:
# Google Drive 마운트 후 저장
from google.colab import drive
drive.mount("/content/drive")

import shutil
save_dir = "/content/drive/MyDrive/finz_model/"
!mkdir -p "{save_dir}"
shutil.copy("models/best/model.pt", save_dir)
print(f"✅ 모델이 Google Drive에 저장됨: {save_dir}model.pt")

Mounted at /content/drive
✅ 모델이 Google Drive에 저장됨: /content/drive/MyDrive/finz_model/model.pt


---

## 9. 모델 평가 (Comprehensive Evaluation)

학습된 모델이 **실제로 유용한지** 다음 항목으로 측정합니다.

| 평가 항목 | 내용 |
|-----------|------|
| 정량 지표 | Accuracy, F1 (macro/weighted), per-class precision/recall |
| Confusion Matrix | 클래스 혼동 패턴 시각화 |
| Calibration | confidence 점수 신뢰성 (예측 확률 ≈ 실제 정확도?) |
| 회귀 품질 | 예측 수익률 vs 실제 수익률 상관관계 |
| 섹터별 성능 | 어떤 섹터에서 잘 맞나 |
| 베이스라인 비교 | 무작위/단순 다수결 대비 얼마나 나은가 |

In [14]:
##── 9-0. 환경 셋업 ────────────────────────────────────────────────
# (학습 직후 이 노트북을 이어서 실행하는 경우 이미 import 완료)

import torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)
from torch.utils.data import DataLoader
from pathlib import Path

# ── 모델 / config 불러오기 ─────────────────────────────────────────
# (Drive에서 불러올 경우 경로를 아래처럼 지정)
MODEL_PATH = "models/best/model.pt"
# MODEL_PATH = "/content/drive/MyDrive/finz_model/model.pt"  # Drive 사용 시

from src.config import load_config
from src.models.momentum_model import FinzMomentumModel
from src.data.dataset import FinzNewsDataset

cfg = load_config()
DATA_PATH = "data/processed/dataset.parquet"
HORIZONS = cfg["data"]["horizons"]          # [30, 180, 360]
LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
model = FinzMomentumModel(
    num_numerical_features=ckpt["num_numerical_features"],
    text_model_name=cfg["text_encoder"]["model_name"],
    text_embed_dim=cfg["model"]["text_embed_dim"],
    numerical_hidden=cfg["model"]["numerical_hidden"],
    fusion_hidden=cfg["model"]["fusion_hidden"],
    num_heads=cfg["model"]["num_heads"],
    dropout=0.0,
    num_horizons=cfg["model"]["num_horizons"],
)
model.load_state_dict(ckpt["model_state_dict"])
model.eval().to(DEVICE)

test_ds = FinzNewsDataset(DATA_PATH, cfg=cfg, split="test")
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)
print(f"Test samples: {len(test_ds)} | Device: {DEVICE}")
print(f"Best epoch: {ckpt['epoch']}  |  Best val_loss: {ckpt['val_loss']:.4f}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Test samples: 108 | Device: cuda
Best epoch: 7  |  Best val_loss: 1.9728


In [15]:
##── 9-1. 테스트셋 전체 추론 ────────────────────────────────────────

all_labels  = [[] for _ in HORIZONS]   # 실제 레이블
all_preds   = [[] for _ in HORIZONS]   # 예측 레이블
all_probs   = [[] for _ in HORIZONS]   # softmax 확률 (3-class)
all_returns_true = [[] for _ in HORIZONS]
all_returns_pred = [[] for _ in HORIZONS]
symbols_list = []

# 심볼 컬럼이 test_ds에 있으면 수집
df_full = pd.read_parquet(DATA_PATH).sort_values("published_at").reset_index(drop=True)
n = len(df_full)
val_ratio  = cfg["training"]["val_split"]
test_ratio = cfg["training"]["test_split"]
val_end  = int(n * (1 - test_ratio))
df_test = df_full.iloc[val_end:].reset_index(drop=True)

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        ids   = batch["input_ids"].to(DEVICE)
        mask  = batch["attention_mask"].to(DEVICE)
        num   = batch["numerical"].to(DEVICE)
        lbls  = batch["labels"]    # (B, num_horizons)
        rets  = batch["returns"]   # (B, num_horizons)

        outputs = model(ids, mask, num)

        for i in range(len(HORIZONS)):
            logits = outputs[f"logits_{i}"].cpu()
            probs  = torch.softmax(logits, dim=-1)
            preds  = probs.argmax(dim=-1)

            all_labels[i].extend(lbls[:, i].tolist())
            all_preds[i].extend(preds.tolist())
            all_probs[i].extend(probs.tolist())
            all_returns_true[i].extend(rets[:, i].tolist())
            all_returns_pred[i].extend(outputs[f"return_{i}"].cpu().tolist())

print("추론 완료 ✅")

추론 완료 ✅


In [17]:
##── 9-2. 정량 지표 (Accuracy / F1 / Classification Report) ─────────

print("=" * 65)
print(f"{'HORIZON':<10} {'ACC':>7} {'F1-macro':>10} {'F1-weighted':>13}")
print("=" * 65)
for i, h in enumerate(HORIZONS):
    y_true = all_labels[i]
    y_pred = all_preds[i]
    acc    = accuracy_score(y_true, y_pred)
    f1m    = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    f1w    = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    print(f"{h:>4}d       {acc:>7.3f} {f1m:>10.3f} {f1w:>13.3f}")
print("=" * 65)

print("\n▶ 상세 리포트 (30d horizon):")
y_true_30d = all_labels[0]
y_pred_30d = all_preds[0]
unique_labels_30d = sorted(list(set(y_true_30d)))
all_target_names = ["negative", "neutral", "positive"]
display_target_names_30d = [all_target_names[lbl] for lbl in unique_labels_30d]

print(classification_report(
    y_true_30d, y_pred_30d,
    labels=unique_labels_30d, # Specify which labels are present
    target_names=display_target_names_30d, # Use filtered target names
    zero_division=0,
))

# ── 베이스라인 비교 ───────────────────────────────────────────────
from sklearn.dummy import DummyClassifier
print("─" * 40)
print("베이스라인 (30d, 비교용)")
y_true_30 = all_labels[0]
dummy_uniform  = DummyClassifier(strategy="uniform").fit([[0]]*len(y_true_30), y_true_30)
dummy_majority = DummyClassifier(strategy="most_frequent").fit([[0]]*len(y_true_30), y_true_30)
for name, clf in [("Random guess", dummy_uniform), ("Majority class", dummy_majority)]:
    yp = clf.predict([[0]]*len(y_true_30))
    acc = accuracy_score(y_true_30, yp)
    f1  = f1_score(y_true_30, yp, average="macro", zero_division=0)
    print(f"  {name:<18}: acc={acc:.3f}  f1-macro={f1:.3f}")

HORIZON        ACC   F1-macro   F1-weighted
  30d         1.000      1.000         1.000
 180d         0.954      0.488         0.976
 360d         0.907      0.317         0.951

▶ 상세 리포트 (30d horizon):
              precision    recall  f1-score   support

     neutral       1.00      1.00      1.00       108

    accuracy                           1.00       108
   macro avg       1.00      1.00      1.00       108
weighted avg       1.00      1.00      1.00       108

────────────────────────────────────────
베이스라인 (30d, 비교용)
  Random guess      : acc=1.000  f1-macro=1.000
  Majority class    : acc=1.000  f1-macro=1.000


In [ ]:
##── 9-3. Confusion Matrix (3개 horizon) ────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, h in enumerate(HORIZONS):
    cm = confusion_matrix(all_labels[i], all_preds[i], labels=[0, 1, 2])
    disp = ConfusionMatrixDisplay(cm, display_labels=["neg", "neu", "pos"])
    disp.plot(ax=axes[i], colorbar=False, cmap="Blues")
    axes[i].set_title(f"{h}d Horizon", fontsize=13)

plt.suptitle("Confusion Matrix — Test Set", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
##── 9-4. Confidence Calibration (신뢰도 캘리브레이션) ──────────────
# 모델이 "80% 확신"이라고 할 때 실제로 80%를 맞추는가?

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
N_BINS = 10
bin_edges = np.linspace(0, 1, N_BINS + 1)

for i, h in enumerate(HORIZONS):
    probs_np = np.array(all_probs[i])
    labels_np = np.array(all_labels[i])
    preds_np  = np.array(all_preds[i])
    max_conf  = probs_np.max(axis=1)        # 예측 클래스 confidence
    correct   = (preds_np == labels_np).astype(float)

    bin_acc  = []
    bin_conf = []
    bin_cnt  = []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (max_conf >= lo) & (max_conf < hi)
        if mask.sum() > 0:
            bin_acc.append(correct[mask].mean())
            bin_conf.append(max_conf[mask].mean())
            bin_cnt.append(mask.sum())

    ax = axes[i]
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect calibration")
    ax.plot(bin_conf, bin_acc, "o-", label="Model")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
    ax.set_title(f"{h}d Calibration")
    ax.legend(fontsize=8)

    # ECE (Expected Calibration Error)
    ece = sum(abs(a - c) * n for a, c, n in zip(bin_acc, bin_conf, bin_cnt)) / max(len(labels_np), 1)
    ax.text(0.05, 0.92, f"ECE={ece:.3f}", transform=ax.transAxes, fontsize=9,
            color="red" if ece > 0.1 else "green")

plt.suptitle("Confidence Calibration (낮을수록 좋음 — ECE < 0.1 이면 양호)", fontsize=13)
plt.tight_layout()
plt.show()
print("ECE (Expected Calibration Error): 0이면 완벽, 0.1 이하면 양호, 0.2 이상이면 recalibration 필요")

In [ ]:
##── 9-5. 회귀 품질 (예측 수익률 vs 실제 수익률) ─────────────────────
from scipy.stats import pearsonr, spearmanr

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, h in enumerate(HORIZONS):
    y_true_r = np.array(all_returns_true[i])
    y_pred_r = np.array(all_returns_pred[i])

    # 이상치 제거 (±50% 초과는 시각화에서 제외)
    mask = (np.abs(y_true_r) < 50) & (np.abs(y_pred_r) < 50)
    yt, yp = y_true_r[mask], y_pred_r[mask]

    pear_r, _ = pearsonr(yt, yp) if len(yt) > 2 else (0, 1)
    spear_r, _ = spearmanr(yt, yp) if len(yt) > 2 else (0, 1)
    mae = np.abs(yt - yp).mean()

    ax = axes[i]
    ax.scatter(yt, yp, alpha=0.4, s=15, c="steelblue")
    lim = max(np.abs(yt).max(), np.abs(yp).max(), 1)
    ax.plot([-lim, lim], [-lim, lim], "r--", alpha=0.6)
    ax.set_xlabel("Actual return (%)")
    ax.set_ylabel("Predicted return (%)")
    ax.set_title(f"{h}d Return Regression")
    ax.text(0.05, 0.92,
            f"Pearson r={pear_r:.2f}\nSpearman r={spear_r:.2f}\nMAE={mae:.2f}%",
            transform=ax.transAxes, fontsize=8, verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.7))

plt.suptitle("Predicted vs Actual Return (%) — 대각선에 가까울수록 좋음", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
##── 9-6. 섹터별 정확도 (30d) ─────────────────────────────────────────

# 섹터 매핑
SECTOR_MAP = {
    "TSLA":"Tech/AI","NVDA":"Tech/AI","GOOGL":"Tech/AI","PLTR":"Tech/AI",
    "AAPL":"Tech/AI","IONQ":"Tech/AI","MSFT":"Tech/AI","AMZN":"Tech/AI",
    "ASML":"Tech/AI","ORCL":"Tech/AI","SMR":"Tech/AI","INTC":"Tech/AI","IREN":"Tech/AI",
    "JPM":"Finance","GS":"Finance","V":"Finance","BRK-B":"Finance",
    "XOM":"Energy","CVX":"Energy","NEE":"Energy","ENPH":"Energy",
    "JNJ":"Healthcare","PFE":"Healthcare","MRNA":"Healthcare","ABBV":"Healthcare",
    "KO":"Consumer","PEP":"Consumer","MCD":"Consumer","PG":"Consumer",
    "NKE":"Retail","TJX":"Retail","LULU":"Retail",
    "LMT":"Defense","RTX":"Defense","NOC":"Defense",
    "UPS":"Transport","FDX":"Transport","DAL":"Transport",
    "FCX":"Materials","NEM":"Materials","LIN":"Materials",
    "TXN":"Semiconductors","ADI":"Semiconductors","MCHP":"Semiconductors",
}

if "symbol" in df_test.columns:
    df_eval = df_test.copy().iloc[:len(all_labels[0])]
    df_eval["pred_30d"]    = all_preds[0][:len(df_eval)]
    df_eval["correct_30d"] = (df_eval["pred_30d"] == df_eval["label_30d"]).astype(int)
    df_eval["sector"]      = df_eval["symbol"].map(SECTOR_MAP).fillna("Other")

    sector_acc = (df_eval.groupby("sector")["correct_30d"]
                  .agg(["mean", "count"])
                  .rename(columns={"mean": "accuracy", "count": "n_samples"})
                  .sort_values("accuracy", ascending=True))

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ["#c0392b" if v < 0.4 else "#e67e22" if v < 0.55 else "#27ae60"
              for v in sector_acc["accuracy"]]
    bars = ax.barh(sector_acc.index, sector_acc["accuracy"], color=colors)
    ax.axvline(1/3, color="gray", linestyle="--", alpha=0.7, label="Random (33.3%)")
    ax.axvline(sector_acc["accuracy"].mean(), color="navy", linestyle=":", alpha=0.8, label="Overall mean")
    for bar, n in zip(bars, sector_acc["n_samples"]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f"n={n}", va="center", fontsize=8)
    ax.set_xlabel("Test Accuracy (30d)")
    ax.set_title("Sector-level Accuracy — 30d Horizon")
    ax.legend(); plt.tight_layout(); plt.show()

    print(sector_acc.to_string())
else:
    print("symbol 컬럼이 없어 섹터 분석을 건너뜁니다.")

In [ ]:
##── 9-7. 고신뢰도 필터링 효과 ─────────────────────────────────────
# "확신도 높은 예측만 필터링하면 정확도가 올라가는가?"
# 실제 서비스에서는 confidence_pct 임계값으로 예측 품질을 높일 수 있음

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
THRESHOLDS = np.arange(0.33, 0.95, 0.05)

for i, h in enumerate(HORIZONS):
    probs_np  = np.array(all_probs[i])
    labels_np = np.array(all_labels[i])
    preds_np  = np.array(all_preds[i])
    max_conf  = probs_np.max(axis=1)
    correct   = (preds_np == labels_np)

    accs, coverages = [], []
    for thr in THRESHOLDS:
        mask = max_conf >= thr
        if mask.sum() == 0:
            break
        accs.append(correct[mask].mean())
        coverages.append(mask.mean())

    ax = axes[i]
    ax2 = ax.twinx()
    ax.plot(THRESHOLDS[:len(accs)], accs, "o-", color="steelblue", label="Accuracy")
    ax2.plot(THRESHOLDS[:len(accs)], [c*100 for c in coverages], "s--", color="orange", label="Coverage %")
    ax.set_xlabel("Confidence threshold")
    ax.set_ylabel("Accuracy", color="steelblue")
    ax2.set_ylabel("Coverage (%)", color="orange")
    ax.set_title(f"{h}d — Confidence Filtering")
    ax.set_ylim(0, 1)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

plt.suptitle("High-confidence filtering: 임계값 높일수록 정확도↑ 커버리지↓", fontsize=12)
plt.tight_layout()
plt.show()

## 📊 평가 결과 해석 가이드

### 지표별 판독 기준

| 지표 | 양호 | 주의 | 불량(재학습 필요) |
|------|------|------|-------------------|
| **Accuracy (30d)** | > 55% | 45~55% | < 45% (랜덤 33%) |
| **F1-macro (30d)** | > 0.45 | 0.33~0.45 | < 0.33 |
| **ECE (캘리브레이션)** | < 0.10 | 0.10~0.20 | > 0.20 |
| **Pearson r (회귀)** | > 0.3 | 0.1~0.3 | < 0.1 |
| **Conf 80% 이상 Acc** | > 70% | 55~70% | < 55% |

### 공통 문제 패턴과 해결책

| 현상 | 원인 추정 | 해결 방법 |
|------|----------|-----------|
| Neutral 쪽으로 과도하게 예측 | 클래스 불균형 | `WeightedRandomSampler` 또는 class weight 추가 |
| Train-Val 격차 큼, Test 낮음 | Overfitting | Dropout 증가, `freeze_layers` 늘리기, 데이터 추가 |
| ECE 높음 (과신) | 모델이 지나치게 확신 | Temperature Scaling 적용 |
| 회귀 Pearson r ≈ 0 | 수익률 예측이 무의미 | 레이블 임계값 조정, 더 많은 데이터 필요 |
| 360d가 30d보다 나쁨 | 장기 예측 노이즈 | 정상이므로 무시 가능 |

### 현재 데이터 규모에 대한 기대치
Yahoo Finance만으로는 종목당 ~10개 기사 → 총 ~400개 샘플로 **테스트셋이 ~40개** 수준입니다.
이 규모에서는 통계적 신뢰성이 낮으므로, **Financial PhraseBank + Kaggle CSV 추가 후 재학습을 권장**합니다.